# What & why

This notebook provides a function that adds a number of text features to all of the CommonLit Readability challenge dataset.

Depending on your model, it might be worth trying out including those features. Although they are comparatively rough in contrast to more sophisticated methods such as transformer-based models, some of these metrics are correlated with excerpt complexity. In addition, adding a model based on them to your prediction ensemble might improve performance.

# How

This function is based on the package `textstat`. It provides a very easy-to-use API to extract a number of features from texts. For more details on these metrics, please refer to their documentation at https://textstat.readthedocs.io/en/latest/.

# How to use

You can either: 
1) import the dataset into your inference notebook; or
2) use the function `include_text_features` in your dataset. 

If you choose (1), please consider using a good validation scheme. If you choose (2), pass the function separately to your training, validation and test datasets could also help with performance.

Also, if you pass the values to a neural network, consider subtracting the batch mean and dividing by the batch standard deviation, since neural networks prefer working on normalised data.

# Ok, that's it...

Good luck in the competition, and leave your comment below with feedback and with your impressions from using these features!

# Code

In [1]:
!pip install textstat

     |████████████████████████████████| 99 kB 3.4 MB/s 
     |████████████████████████████████| 1.9 MB 8.0 MB/s 


In [2]:
import pandas as pd
import textstat

In [3]:
file_path = "../input/commonlitreadabilityprize/"
train_df = pd.read_csv(file_path+"train.csv")
train_df = train_df[['id', 'excerpt', 'target']]
train_df = train_df.sample(frac=1).reset_index(drop=True)

In [4]:
def text_features(excerpt):
    feat = excerpt.map(lambda x: {
        "flesch_reading_ease": textstat.flesch_reading_ease(x),
        "smog_index": textstat.smog_index(x),
        "flesch_kincaid_grade": textstat.flesch_kincaid_grade(x),
        "coleman_liau_index": textstat.coleman_liau_index(x),
        "automated_readability_index": textstat.automated_readability_index(x),
        "dale_chall_readability_score": textstat.dale_chall_readability_score(x),
        "difficult_words": textstat.difficult_words(x),
        "linsear_write_formula": textstat.linsear_write_formula(x),
        "gunning_fog": textstat.gunning_fog(x),
#        textstat.text_standard(excerpt)
    })
    return pd.DataFrame([pd.Series(x) for x in pd.DataFrame(feat).iloc[:,0]])

def include_text_features(df):
    return pd.concat([df, text_features(df.excerpt)], axis=1)

In [5]:
augmented_df = include_text_features(train_df)
augmented_df

,id,excerpt,target,flesch_reading_ease,smog_index,flesch_kincaid_grade,coleman_liau_index,automated_readability_index,dale_chall_readability_score,difficult_words,linsear_write_formula,gunning_fog
0,3ee9d45f9,"""Oh, father,"" said a little Frog to the big on...",0.008936,78.52,8.0,8.9,4.59,11.0,1.82,6.0,13.250000,11.21
1,ed634f405,The first glimpse of Mont St. Michel is strang...,-1.121302,63.02,12.2,10.7,9.06,12.9,7.72,36.0,12.200000,13.60
2,381bc33b5,Throughout the long and difficult period of Wa...,-1.708766,53.65,14.6,12.2,9.93,13.7,7.89,34.0,21.666667,15.00
3,08104f366,We three reached the old poplar the next eveni...,-1.394485,73.00,9.3,8.9,10.39,13.2,7.38,23.0,13.750000,10.82
4,32e3b2efe,"The primary object of a launch, in the modern ...",-2.348231,40.76,15.9,17.2,11.67,21.5,9.23,45.0,30.000000,20.00
...,...,...,...,...,...,...,...,...,...,...,...,...
2829,b858f1375,"""Oh, wise judge,"" he cried, ""I have come to yo...",0.142540,103.97,4.9,3.2,3.77,6.9,1.07,2.0,4.666667,7.20
2830,f482c02a0,Augmented reality (AR) is a live direct or ind...,-2.251382,23.26,17.3,15.6,13.99,15.8,9.23,51.0,16.800000,16.11
2831,268780694,I will tell you what stopped him. While the du...,-0.623943,96.82,5.6,3.9,4.00,5.5,1.43,7.0,4.666667,6.90
2832,39fde689a,A woman with her baby went into the forest. Sh...,0.568792,97.40,5.2,1.6,2.68,1.7,1.00,7.0,2.714286,3.34


In [6]:
augmented_df.to_csv("augmented_df.csv")